In [ ]:
cell = "GM12878"
resolution = 10000    # Hi-C resolution (bp): 10000(10kb), 25000(25kb)
display_reso = int(resolution / 1000)
root_dir = "/mnt/d/TAD-MultiOmicsNet"

In [ ]:
### Generate masks

In [ ]:
import numpy as np
import os

RAW_DIR = f"{root_dir}/dnabert/all_chr_raw/"
OUT_MASK_DIR = f"{root_dir}/dnabert/all_chr_mask/"
os.makedirs(OUT_MASK_DIR, exist_ok=True)


def get_nonzero_mask(arr: np.ndarray) -> np.ndarray:
    #Return 0/1 one-dimensional mask: 1 = normal base; 0 = all-zero bin for N bases
    return (~(np.max(np.abs(arr), axis=1) < 1e-7)).astype(np.float32)

if __name__ == "__main__":
    print("Start generating the DNABERT NAN mask file\n")

    for fname in os.listdir(RAW_DIR):
        if not fname.endswith("_all_1kb_feat.npy"):
            continue

        raw_path = os.path.join(RAW_DIR, fname)
        raw_arr = np.load(raw_path).astype(np.float32)
        mask_1d = get_nonzero_mask(raw_arr)

        mask_out_name = fname.replace("_all_1kb_feat.npy", "_mask.npy")
        mask_save_path = os.path.join(OUT_MASK_DIR, mask_out_name)

        np.save(mask_save_path, mask_1d)
        print(f"{fname:45s} | mask shape:{mask_1d.shape}, N-bin count：{(mask_1d == 0).sum()}")

    print("\n✅All chromosome masks have been generated.")

In [ ]:
### dimension reduction

In [ ]:
import numpy as np
import os
import joblib
from sklearn.decomposition import IncrementalPCA


TARGET_DIM = 64
RAW_FEAT_DIR = f"{root_dir}/dnabert/all_chr_raw/"
OUT_PCA_DIR = f"{root_dir}/dnabert/all_chr_pca{TARGET_DIM}/"
os.makedirs(OUT_PCA_DIR, exist_ok=True)

BATCH_SIZE = 50000
train_chroms = [f"chr{i}" for i in range(1,13)]

all_npy_files = sorted([f for f in os.listdir(RAW_FEAT_DIR) if f.endswith("_all_1kb_feat.npy")])

# Step1：PCA fitting using only training set chromosomes
ipca = IncrementalPCA(n_components=TARGET_DIM, batch_size=BATCH_SIZE)
print("=== PCA fitting using only training set chromosomes ===")
for fname in all_npy_files:
    chr_id = fname.replace("_all_1kb_feat.npy", "")
    if chr_id not in train_chroms:
        continue
    arr = np.load(os.path.join(RAW_FEAT_DIR, fname))
    ipca.partial_fit(arr)
    print(f"fit: {fname}, shape {arr.shape}")
    del arr

sum_var = np.sum(ipca.explained_variance_ratio_)
print(f"Fitting completed, target dimension {TARGET_DIM}, total explained variance {sum_var:.3f}")
joblib.dump(ipca, os.path.join(OUT_PCA_DIR, f"ipca_{TARGET_DIM}.pkl"))
print(f"ipca_{TARGET_DIM}.pkl is saved to {OUT_PCA_DIR}")

# Step2：Perform IPCA obtained from the training set to transform all chromosomes (including val/test chromosomes) 
print("\n=== transform all chromosomes ===")
for fname in all_npy_files:
    arr = np.load(os.path.join(RAW_FEAT_DIR, fname))
    arr_pca = ipca.transform(arr).astype(np.float32)
    out_name = fname.replace("_all_1kb_feat.npy", f"_all_1kb_feat_pca{TARGET_DIM}.npy")
    out_path = os.path.join(OUT_PCA_DIR, out_name)
    np.save(out_path, arr_pca)
    print(f"{fname:40s} → {out_name}, shape {arr_pca.shape}")
    del arr, arr_pca

print("All chromosome dimension reduction completed")


In [ ]:
### Generate upstream and downstream data

In [ ]:

import os
import numpy as np
import pandas as pd

reso = display_reso
SEED_LIST = [42, 43, 44, 45, 46]

DNA_RAW_DIR = f"{root_dir}/dnabert/all_chr_pca64/" 
DNA_MASK_DIR = f"{root_dir}/dnabert/all_chr_mask/" 
FEAT_DIM = 64

def process_single_task(task_info: dict):
    task_name = task_info["task_name"]
    csv_path = task_info["csv_input"]
    dna_out_path = task_info["npy_dna_out"]
    mask_out_path = task_info["npy_mask_out"]
    
    print("\n===================================================")
    print(f"Start processing task: {task_name}")
    print(f"Csv input path: {csv_path}")
    print(f"DNABERT output NPY：{dna_out_path}")
    print(f"Mask output NPY：{mask_out_path}")

    if not os.path.isfile(csv_path):
        raise FileNotFoundError(f"[{task_name}] The input CSV does not exist. Please check the path: {csv_path}")
    for outp in [dna_out_path, mask_out_path]:
        if os.path.exists(outp):
            print(f"Warning: The output file {outp} already exists and will be overwritten directly in this run!")

    # Automatically create the output directory
    output_dir = os.path.dirname(dna_out_path)
    os.makedirs(output_dir, exist_ok=True)

    df = pd.read_csv(csv_path)
    chr_all = df["chr"].values
    start_all = df["start"].values
    sample_count = len(df)
    print(f"Total number of current task samples：{sample_count}")

    unique_chrs = sorted(np.unique(chr_all),
                         key=lambda x: int(x.replace("chr", "")) if x.replace("chr", "").isdigit() else (23 if x == "chrX" else 24))
    chr_cache = {}
    chr_mask_cache = {}
    for chrom in unique_chrs:
        chrom_npy = os.path.join(DNA_RAW_DIR, f"{chrom}_all_1kb_feat_pca64.npy")
        chrom_mask_npy = os.path.join(DNA_MASK_DIR, f"{chrom}_mask.npy")
        if not os.path.exists(chrom_npy):
            raise FileNotFoundError(f"Chromosome DNABERT feature missing：{chrom_npy}")
        if not os.path.exists(chrom_mask_npy):
            raise FileNotFoundError(f"Chromosome mask missing：{chrom_mask_npy}")

        chrom_data = np.load(chrom_npy).astype(np.float32)
        chrom_mask = np.load(chrom_mask_npy).astype(np.float32)

        # Dimension verification
        if chrom_data.ndim != 2 or chrom_data.shape[1] != FEAT_DIM:
            raise ValueError(f"{chrom} feature dimension anomaly：{chrom_data.shape}，It is expected that the second dimension is {FEAT_DIM}")
        if chrom_data.shape[0] != chrom_mask.shape[0]:
            raise ValueError(f"{chrom} feature rows ({chrom_data.shape[0]}) != mask rows ({chrom_mask.shape[0]})")

        chr_cache[chrom] = chrom_data
        chr_mask_cache[chrom] = chrom_mask
        print(f"Load {chrom} | feat shape={chrom_data.shape}, mask shape={chrom_mask.shape}")


    output_windows = []
    output_mask_windows = []
    seg_length = 1000
    n_seg_total = 100
    for idx in range(sample_count):
        curr_chr = chr_all[idx]
        curr_start = start_all[idx]
        bin_end = curr_start + resolution
        # Calculate the bin bp center
        center = curr_start + (bin_end - curr_start) / 2
        win_left = int(center - 50000)

        feat_list = []
        mask_list = []
        for seg_idx in range(n_seg_total):
            seg_l = win_left + seg_idx * seg_length
            bin_index = int(seg_l // seg_length) 

            if 0 <= bin_index < chr_cache[curr_chr].shape[0]:
                feat_list.append(chr_cache[curr_chr][bin_index])
                mask_list.append(chr_mask_cache[curr_chr][bin_index])
            else:
                # Chromosome boundary out-of-bounds: set features to all zeros, mask=0
                feat_list.append(np.zeros(FEAT_DIM, dtype=np.float32))
                mask_list.append(0.0)

        output_windows.append(np.stack(feat_list, axis=0))
        output_mask_windows.append(np.array(mask_list))

    final_dna = np.stack(output_windows, axis=0)
    final_mask = np.stack(output_mask_windows, axis=0)  # shape [N_sample,100]

    np.save(dna_out_path, final_dna)
    np.save(mask_out_path, final_mask)

    print(f"{task_name} processing completed!")
    print(f"DNABERT feature array dimension：{final_dna.shape}  [N,100,{FEAT_DIM}]")
    print(f"Mask array dimension：{final_mask.shape}  [N,100]  (1 = valid base bin, 0 = N base / boundary out of bounds)")
    del df, chr_cache, chr_mask_cache, output_windows, output_mask_windows, final_dna, final_mask


# Execute all tasks in batches in a loop
if __name__ == "__main__":
    # Step 1: Iterate over all seeds and generate only the training set
    for SEED in SEED_LIST:
        print(f"==== Step 1：training set, SEED = {SEED} ====")
        train_task = {
            "task_name": "training set",
            "csv_input": f"{root_dir}/{cell}/{reso}kb/samples-generate/train/hic_train_coords_{reso}kb_seed{SEED}.csv",
            "npy_dna_out": f"{root_dir}/{cell}/{reso}kb/samples-generate/train/dna{cell}_train_{reso}kb_seed{SEED}.npy",
            "npy_mask_out": f"{root_dir}/{cell}/{reso}kb/samples-generate/train/dna{cell}_train_mask_{reso}kb_seed{SEED}.npy"
        }
        process_single_task(train_task)
        print(f"SEED{SEED} Training set completed")
    
    # Step 2: Validation Set and Test Set
    print("==== Step 2: Validation Set and Test Set ====")
    val_test_configs = [
        {
            "task_name": "validation set",
            "csv_input": f"{root_dir}/{cell}/{reso}kb/samples-generate/validation/hic_val_coords13-19_{reso}kb.csv",
            "npy_dna_out": f"{root_dir}/{cell}/{reso}kb/samples-generate/validation/dna{cell}_validation_{reso}kb.npy",
            "npy_mask_out": f"{root_dir}/{cell}/{reso}kb/samples-generate/validation/dna{cell}_validation_mask_{reso}kb.npy"
        },
        {
            "task_name": "test set",
            "csv_input": f"{root_dir}/{cell}/{reso}kb/samples-generate/predict/hic_predict_coords20-X_{reso}kb.csv",
            "npy_dna_out": f"{root_dir}/{cell}/{reso}kb/samples-generate/predict/dna{cell}_predict_{reso}kb.npy",
            "npy_mask_out": f"{root_dir}/{cell}/{reso}kb/samples-generate/predict/dna{cell}_predict_mask_{reso}kb.npy"
        }
    ]
    for task in val_test_configs:
        process_single_task(task)
        
    print("The validation set and test set have been generated.")
    print("All tasks completed!")
